# RiskNot: Explainable Credit Default Risk Modeling in Colab

This notebook is a Colab-friendly end-to-end study for the Taiwan credit card default dataset. It covers data cleaning, feature engineering, model training, evaluation, threshold selection, and SHAP explainability.

## How to use in Colab
1. Upload `UCI_Credit_Card.csv` or mount Google Drive.
2. Run the setup cell.
3. Execute cells top to bottom.
4. Save outputs, plots, and trained artifacts to Google Drive if needed.

In [ ]:
# If running in Colab, install dependencies first.
!pip -q install pandas numpy scikit-learn matplotlib seaborn shap lightgbm xgboost joblib imbalanced-learn

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, precision_recall_curve, roc_curve, brier_score_loss)

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except Exception:
    LIGHTGBM_AVAILABLE = False

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

The notebook expects `UCI_Credit_Card.csv` to be available in the Colab runtime. If you are using Google Drive, update the path accordingly.

In [ ]:
# Works with the extracted CSV locally or the original uploaded zip in Colab.
candidate_paths = [
    'Data/UCI_Credit_Card.csv',
    'data/raw/UCI_Credit_Card.csv',
    'UCI_Credit_Card.csv',
    'UCI_Credit_Card.csv.zip',
    '/content/UCI_Credit_Card.csv',
    '/content/UCI_Credit_Card.csv.zip',
]
DATA_PATH = next((path for path in candidate_paths if os.path.exists(path)), None)
if DATA_PATH is None:
    raise FileNotFoundError('Place UCI_Credit_Card.csv or UCI_Credit_Card.csv.zip in the notebook folder, Data/, or /content/.')

df = pd.read_csv(DATA_PATH, compression='zip' if DATA_PATH.endswith('.zip') else 'infer')
df = df.rename(columns={'default.payment.next.month': 'default'})
df = df.drop(columns=['ID'])

print('Shape:', df.shape)
df.head()

## 2. Data Cleaning and Feature Engineering

This section applies the project rules: categorical cleanup, missing-value checks, safe ratio features, and repayment/bill/payment aggregation features.

In [ ]:
def clean_credit_data(dataframe: pd.DataFrame) -> pd.DataFrame:
    cleaned = dataframe.copy()
    cleaned['EDUCATION'] = cleaned['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
    cleaned['MARRIAGE'] = cleaned['MARRIAGE'].replace({0: 3})
    return cleaned

def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype='float64')
    denominator = np.asarray(denominator, dtype='float64')
    result = np.zeros_like(numerator, dtype='float64')
    mask = denominator != 0
    result[mask] = numerator[mask] / denominator[mask]
    result[~np.isfinite(result)] = 0.0
    return np.clip(result, -1000, 1000)

def add_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    df_feat = dataframe.copy()
    pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
    pay_amt_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

    delays = df_feat[pay_cols]
    bills = df_feat[bill_cols]
    payments = df_feat[pay_amt_cols]

    df_feat['max_delay'] = delays.max(axis=1)
    df_feat['avg_delay'] = delays.mean(axis=1)
    df_feat['recent_delay'] = df_feat['PAY_0']
    df_feat['num_delayed_months'] = (delays > 0).sum(axis=1)
    df_feat['severe_delay_flag'] = (delays.max(axis=1) >= 2).astype(int)
    df_feat['repeated_delay_flag'] = (delays.gt(0).sum(axis=1) >= 3).astype(int)
    df_feat['delay_trend'] = df_feat['PAY_0'] - df_feat['PAY_6']

    df_feat['avg_bill_amount'] = bills.mean(axis=1)
    df_feat['max_bill_amount'] = bills.max(axis=1)
    df_feat['min_bill_amount'] = bills.min(axis=1)
    df_feat['bill_amount_std'] = bills.std(axis=1).fillna(0)
    df_feat['recent_bill_amount'] = df_feat['BILL_AMT1']
    df_feat['bill_trend'] = df_feat['BILL_AMT1'] - df_feat['BILL_AMT6']

    df_feat['avg_payment_amount'] = payments.mean(axis=1)
    df_feat['max_payment_amount'] = payments.max(axis=1)
    df_feat['min_payment_amount'] = payments.min(axis=1)
    df_feat['payment_amount_std'] = payments.std(axis=1).fillna(0)
    df_feat['recent_payment_amount'] = df_feat['PAY_AMT1']
    df_feat['payment_trend'] = df_feat['PAY_AMT1'] - df_feat['PAY_AMT6']

    bill_safe = df_feat['avg_bill_amount'].replace(0, np.nan)
    recent_bill_safe = df_feat['BILL_AMT1'].replace(0, np.nan)

    df_feat['credit_utilization_ratio'] = safe_divide(df_feat['avg_bill_amount'], df_feat['LIMIT_BAL'])
    df_feat['recent_utilization_ratio'] = safe_divide(df_feat['BILL_AMT1'], df_feat['LIMIT_BAL'])
    df_feat['payment_to_bill_ratio'] = safe_divide(df_feat['avg_payment_amount'], bill_safe.fillna(0))
    df_feat['recent_payment_to_bill_ratio'] = safe_divide(df_feat['PAY_AMT1'], recent_bill_safe.fillna(0))
    df_feat['low_payment_flag'] = (df_feat['payment_to_bill_ratio'] < 0.2).astype(int)
    df_feat['high_utilization_flag'] = (df_feat['credit_utilization_ratio'] > 0.7).astype(int)
    df_feat['recent_nonpayment_flag'] = (df_feat['PAY_AMT1'] == 0).astype(int)

    df_feat = df_feat.replace([np.inf, -np.inf], np.nan).fillna(0)
    return df_feat

df_clean = clean_credit_data(df)
df_feat = add_features(df_clean)
df_feat.head()

## 3. EDA

The cells below generate core exploratory diagnostics and save the figures locally in the Colab runtime.

In [ ]:
os.makedirs('figures', exist_ok=True)

print('Dataset shape:', df_feat.shape)
print('Missing values:\n', df_feat.isna().sum().sort_values(ascending=False).head(10))
print('Target rate:', df_feat['default'].mean().round(4))
df_feat['default'].value_counts(normalize=True)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='default', data=df_feat, ax=ax[0])
ax[0].set_title('Target Distribution')
sns.histplot(df_feat[df_feat['default'] == 0]['LIMIT_BAL'], color='steelblue', label='No Default', kde=True, ax=ax[1], stat='density', bins=30)
sns.histplot(df_feat[df_feat['default'] == 1]['LIMIT_BAL'], color='tomato', label='Default', kde=True, ax=ax[1], stat='density', bins=30)
ax[1].set_title('LIMIT_BAL by Default Class')
ax[1].legend()
plt.tight_layout()
plt.savefig('figures/target_limit_bal.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Train / Validation / Test Split

We use a stratified 70/15/15 split to avoid leakage and preserve class balance.

In [ ]:
X = df_feat.drop(columns=['default'])
y = df_feat['default']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.mean().round(4), y_val.mean().round(4), y_test.mean().round(4))

## 5. Preprocessing and Models

The preprocessing pipeline keeps tree-based models on numeric engineered features and uses scaling only for logistic regression. Categorical variables are one-hot encoded.

In [ ]:
categorical_features = ['SEX', 'EDUCATION', 'MARRIAGE']
numeric_features = [col for col in X_train.columns if col not in categorical_features]

numeric_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocessor, numeric_features),
        ('cat', categorical_preprocessor, categorical_features),
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', categorical_preprocessor, categorical_features),
    ]
)

models = {
    'logistic_regression': Pipeline(steps=[
        ('preprocessor', preprocessor_lr),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'random_forest': Pipeline(steps=[
        ('preprocessor', preprocessor_tree),
        ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
    ])
}

if LIGHTGBM_AVAILABLE:
    scale_pos_weight = (y_train.value_counts()[0] / y_train.value_counts()[1])
    models['lightgbm'] = Pipeline(steps=[
        ('preprocessor', preprocessor_tree),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE,
            class_weight=None,
            scale_pos_weight=scale_pos_weight,
            n_estimators=300
        ))
    ])

if XGBOOST_AVAILABLE:
    scale_pos_weight = (y_train.value_counts()[0] / y_train.value_counts()[1])
    models['xgboost'] = Pipeline(steps=[
        ('preprocessor', preprocessor_tree),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE,
            n_estimators=300,
            eval_metric='logloss',
            scale_pos_weight=scale_pos_weight,
            tree_method='hist'
        ))
    ])

models.keys()

In [ ]:
results = []
fitted_models = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    val_proba = pipeline.predict_proba(X_val)[:, 1]
    val_pred = (val_proba >= 0.40).astype(int)
    metrics = {
        'model': name,
        'roc_auc': roc_auc_score(y_val, val_proba),
        'pr_auc': average_precision_score(y_val, val_proba),
        'precision': precision_score(y_val, val_pred, zero_division=0),
        'recall': recall_score(y_val, val_pred, zero_division=0),
        'f1': f1_score(y_val, val_pred, zero_division=0),
    }
    results.append(metrics)
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False)
results_df

## 6. Threshold Optimization

The default threshold is not assumed to be 0.50. We compare several cutoffs and select one with a credit-risk interpretation that favors catching risky borrowers.

In [ ]:
final_candidate_name = results_df.iloc[0]['model']
final_model = fitted_models[final_candidate_name]
val_proba = final_model.predict_proba(X_val)[:, 1]
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.60]
threshold_rows = []
for threshold in thresholds:
    pred = (val_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_val, pred, zero_division=0),
        'recall': recall_score(y_val, pred, zero_division=0),
        'f1': f1_score(y_val, pred, zero_division=0),
        'false_positives': fp,
        'false_negatives': fn
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df

In [ ]:
selected_threshold = float(threshold_df.sort_values(['recall', 'f1'], ascending=False).iloc[0]['threshold'])
selected_threshold

## 7. Test Evaluation

Use the selected threshold only after validation-based selection.

In [ ]:
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_threshold).astype(int)

print('ROC-AUC:', roc_auc_score(y_test, test_proba).round(4))
print('PR-AUC:', average_precision_score(y_test, test_proba).round(4))
print('Brier score:', brier_score_loss(y_test, test_proba).round(4))
print(classification_report(y_test, test_pred, zero_division=0))

cm = confusion_matrix(y_test, test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 8. SHAP Explainability

The code below prepares a SHAP explainer for tree-based models. For local explanations, it shows the strongest risk-increasing and risk-decreasing factors.

In [ ]:
def get_feature_names(preprocessor):
    feature_names = []
    for name, transformer, columns in preprocessor.transformers_:
        if name == 'remainder' and transformer == 'drop':
            continue
        if hasattr(transformer, 'named_steps') and 'onehot' in transformer.named_steps:
            onehot = transformer.named_steps['onehot']
            encoded = onehot.get_feature_names_out(columns)
            feature_names.extend(encoded.tolist())
        elif hasattr(transformer, 'get_feature_names_out'):
            try:
                feature_names.extend(transformer.get_feature_names_out(columns).tolist())
            except Exception:
                feature_names.extend(list(columns))
        else:
            feature_names.extend(list(columns))
    return feature_names

preprocessor = final_model.named_steps['preprocessor']
model = final_model.named_steps['model']
X_test_transformed = preprocessor.transform(X_test)
feature_names = get_feature_names(preprocessor)

if LIGHTGBM_AVAILABLE and final_candidate_name == 'lightgbm':
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_transformed)
    shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.show()

    sample_index = 0
    sample_values = X_test_transformed[sample_index]
    sample_shap = shap_values[1][sample_index] if isinstance(shap_values, list) else shap_values[sample_index]
    top_idx = np.argsort(np.abs(sample_shap))[-10:][::-1]
    explanation = pd.DataFrame({
        'feature': np.array(feature_names)[top_idx],
        'value': np.array(sample_values).reshape(-1)[top_idx],
        'shap_value': sample_shap[top_idx]
    })
    explanation['direction'] = np.where(explanation['shap_value'] >= 0, 'increases risk', 'decreases risk')
    explanation

## 9. Risk Scoring Helper

This converts a probability into the requested score and risk segment.

In [ ]:
def risk_from_probability(default_probability: float):
    risk_score = int(round(default_probability * 100))
    risk_score = max(0, min(100, risk_score))
    if risk_score <= 30:
        segment = 'Low Risk'
    elif risk_score <= 60:
        segment = 'Medium Risk'
    else:
        segment = 'High Risk'
    return {
        'default_probability': float(default_probability),
        'risk_score': risk_score,
        'risk_segment': segment
    }

risk_from_probability(test_proba[0])

## 10. Save Artifacts

Save the fitted model, threshold, and feature metadata for later reuse in a dashboard or script.

In [ ]:
os.makedirs('models', exist_ok=True)
artifacts = {
    'model': final_model,
    'selected_threshold': selected_threshold,
    'feature_names': feature_names,
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
}
joblib.dump(artifacts, 'models/risknot_artifacts.joblib')
print('Saved to models/risknot_artifacts.joblib')

## 11. Next Steps

If you want, I can turn this Colab notebook into a fuller project with separate Python modules, a README, and a Streamlit app.